# Stage 3B - LLM Fine-Tuning (LoRA bf16)
## Model: `Qwen/Qwen2.5-7B-Instruct` | 2-class fine-tune

This notebook fine-tunes `Qwen2.5-7B-Instruct` with **LoRA in bf16 full precision** (no quantization).
Designed for GPUs with ≥ 40GB VRAM (e.g. A100 80GB).

**Differences from the QLoRA version (`stage3_qwen_2class_finetune.ipynb`):**

| Item | QLoRA version | This version (bf16) |
|---|---|---|
| Quantization | 4-bit NF4 | None — full bf16 |
| LoRA r | 16 | 32 |
| LoRA alpha | 32 | 64 |
| Batch size | 4 | 8 |
| Output folder | `*_finetune/` | `*_finetune_bf16/` |

**Task:** Binary sentiment (0 = Negative 1-2★, 1 = Positive 4-5★, 3★ removed)

Saved outputs: predictions CSV, summary JSON, metrics CSV, confusion matrix PNG, LoRA adapter.

In [ ]:
!pip install -q transformers datasets peft trl accelerate \
               pandas scikit-learn matplotlib seaborn tqdm

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
import os
import re
import json
import time
import math
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, mean_squared_error, mean_absolute_error,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ============================================
# Paths and configuration
# ============================================
TRAIN_PATH = "/content/drive/MyDrive/Yelp_Project/data/processed/train_data.csv"
VAL_PATH   = "/content/drive/MyDrive/Yelp_Project/data/processed/val_data.csv"
TEST_PATH  = "/content/drive/MyDrive/Yelp_Project/data/processed/test_data.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Yelp_Project/LLM/qwen2_5_7b_instruct_2class_finetune_bf16"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEXT_COL  = "text"
LABEL_COL = "stars"

TASK_TYPE  = "2_class"
RUN_ID     = "qwen2_5_7b_instruct_2class_finetune_bf16"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# LoRA hyper-parameters (larger r thanks to 80GB VRAM)
LORA_R          = 32
LORA_ALPHA      = 64
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]

# Training hyper-parameters
MAX_SEQ_LEN  = 512   # used during training
TRAIN_EPOCHS = 3
BATCH_SIZE   = 8
GRAD_ACCUM   = 4     # effective batch = 32
LR           = 2e-4
WARMUP_RATIO = 0.05
LR_SCHEDULER = "cosine"

# Inference
# MAX_INFER_LEN is intentionally larger than MAX_SEQ_LEN:
# long reviews exceed MAX_SEQ_LEN tokens and truncate the instruction,
# causing the model to continue the review instead of classifying it.
INFER_BATCH_SIZE = 16
MAX_NEW_TOKENS   = 64
MAX_INFER_LEN    = 1024   # covers ~99% of Yelp reviews without truncating the prompt

LABELS         = [0, 1]
DISPLAY_LABELS = ["Negative (0)", "Positive (1)"]

print(f"Run ID         : {RUN_ID}")
print(f"Model          : {MODEL_NAME}")
print(f"Task           : {TASK_TYPE}")
print(f"MAX_SEQ_LEN    : {MAX_SEQ_LEN}  (training)")
print(f"MAX_INFER_LEN  : {MAX_INFER_LEN}  (inference)")
print(f"GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================
# Load and prepare datasets
# ============================================
def load_and_filter(path, task_type):
    df = pd.read_csv(path)[[TEXT_COL, LABEL_COL]].copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    if task_type == "2_class":
        df = df[df[LABEL_COL] != 3].copy()
        df["original_stars"] = df[LABEL_COL]
        df["label"] = df[LABEL_COL].apply(lambda x: 0 if x <= 2 else 1)
    else:
        df["original_stars"] = df[LABEL_COL]
        df["label"] = df[LABEL_COL]
    return df

df_train = load_and_filter(TRAIN_PATH, TASK_TYPE)
df_val   = load_and_filter(VAL_PATH,   TASK_TYPE)
df_test  = load_and_filter(TEST_PATH,  TASK_TYPE)

print(f"Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}")
print(df_train[[TEXT_COL, 'original_stars', 'label']].head())

In [ ]:
# ============================================
# Prompt helpers
# ============================================
SYSTEM_MSG = (
    "You are a Yelp review sentiment classifier. "
    "Given a review, output 0 for Negative (original 1-2 stars) "
    "or 1 for Positive (original 4-5 stars). "
    "Your response must be exactly one of: Final label: 0  or  Final label: 1"
)

def build_chat_messages(text: str, label=None):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Review: {text}"},
    ]
    if label is not None:
        messages.append({"role": "assistant", "content": f"Final label: {label}"})
    return messages

def parse_output(text: str):
    text = str(text).strip()
    match = re.search(r"Final label:\s*([01])", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1)), True
    binary_matches = re.findall(r"\b([01])\b", text)
    if binary_matches:
        return int(binary_matches[-1]), False
    lowered = text.lower()
    if "negative" in lowered:
        return 0, False
    if "positive" in lowered:
        return 1, False
    return None, False

In [ ]:
# ============================================
# Load tokenizer
# ============================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_SEQ_LEN

print("Vocab size       :", tokenizer.vocab_size)
print("EOS token        :", tokenizer.eos_token)
print("model_max_length :", tokenizer.model_max_length)

In [ ]:
# ============================================
# Build HuggingFace Dataset with chat template
# ============================================
def df_to_hf_dataset(df):
    records = []
    for _, row in df.iterrows():
        messages = build_chat_messages(row[TEXT_COL], row["label"])
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        records.append({"text": formatted})
    return Dataset.from_list(records)

train_dataset = df_to_hf_dataset(df_train)
val_dataset   = df_to_hf_dataset(df_val)

print("Train dataset size:", len(train_dataset))
print("Val   dataset size:", len(val_dataset))
print("\nSample:")
print(train_dataset[0]["text"][:400])

In [ ]:
# ============================================
# Load base model in bf16 (no quantization)
# ============================================
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # full bf16, no 4-bit quantization
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used after model load: {used_gb:.1f} GB")

In [ ]:
# ============================================
# Fine-tune with SFTTrainer
# ============================================
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter")

steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
warmup_steps = max(1, int(WARMUP_RATIO * steps_per_epoch * TRAIN_EPOCHS))
print(f"steps_per_epoch : {steps_per_epoch}")
print(f"warmup_steps    : {warmup_steps}")

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=warmup_steps,
    lr_scheduler_type=LR_SCHEDULER,
    fp16=False,
    bf16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start
print(f"Training finished in {train_time:.1f}s ({train_time/60:.1f} min)")

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to: {ADAPTER_DIR}")

In [ ]:
# ============================================
# Reload model for inference
# ============================================
del model
del base_model
torch.cuda.empty_cache()

base_model_inf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
base_model_inf.config.use_cache = True

inf_model = PeftModel.from_pretrained(base_model_inf, ADAPTER_DIR)
inf_model.eval()
print("Fine-tuned model loaded for inference.")

In [ ]:
# ============================================
# Batched inference on test set
# ============================================
all_rows = []
eval_start = time.time()

for start in tqdm(range(0, len(df_test), INFER_BATCH_SIZE), desc="Inference"):
    end = min(start + INFER_BATCH_SIZE, len(df_test))
    batch = df_test.iloc[start:end]

    prompts = []
    for _, row in batch.iterrows():
        messages = build_chat_messages(row[TEXT_COL], label=None)
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt)

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_INFER_LEN,   # larger than MAX_SEQ_LEN to avoid truncating the instruction
    ).to(inf_model.device)

    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

    for i, raw_text in enumerate(decoded):
        pred, strict_ok = parse_output(raw_text)
        all_rows.append({
            "text"            : batch.iloc[i][TEXT_COL],
            "original_stars"  : int(batch.iloc[i]["original_stars"]),
            "true_label"      : int(batch.iloc[i]["label"]),
            "raw_output"      : raw_text,
            "pred"            : pred,
            "strict_format_ok": strict_ok,
        })

eval_time_seconds = time.time() - eval_start

result_df = pd.DataFrame(all_rows)
pred_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_predictions.csv")
result_df.to_csv(pred_path, index=False, encoding="utf-8")
print(f"Saved predictions to: {pred_path}")
print(f"Eval time (s): {eval_time_seconds:.2f}")
print(result_df.head())

In [ ]:
# ============================================
# Evaluation
# ============================================
eval_df = result_df.dropna(subset=["pred"]).copy()
eval_df["pred"] = eval_df["pred"].astype(int)

y_true = eval_df["true_label"].to_numpy()
y_pred = eval_df["pred"].to_numpy()

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
cm       = confusion_matrix(y_true, y_pred, labels=LABELS)
mse      = mean_squared_error(y_true, y_pred)
mae      = mean_absolute_error(y_true, y_pred)

valid_prediction_rate = result_df["pred"].notna().mean()
strict_format_rate    = result_df["strict_format_ok"].mean()
eval_speed            = len(result_df) / eval_time_seconds if eval_time_seconds > 0 else np.nan

report = classification_report(
    y_true, y_pred, labels=LABELS, output_dict=True, digits=4
)

summary = {
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "LoRA_Finetune_BF16",
    "Model"                : MODEL_NAME,
    "LoRA_r"               : LORA_R,
    "LoRA_alpha"           : LORA_ALPHA,
    "Train_Epochs"         : TRAIN_EPOCHS,
    "Effective_Batch_Size" : BATCH_SIZE * GRAD_ACCUM,
    "Learning_Rate"        : LR,
    "Train_Time(s)"        : float(train_time),
    "Batch_Size"           : INFER_BATCH_SIZE,
    "Accuracy"             : float(accuracy),
    "Macro_F1"             : float(macro_f1),
    "Valid_Prediction_Rate": float(valid_prediction_rate),
    "Strict_Format_Rate"   : float(strict_format_rate),
    "Off_By_1_Acc"         : None,
    "Adj_Error_Ratio"      : None,
    "Mid_Confusion_Ratio"  : None,
    "MSE"                  : float(mse),
    "MAE"                  : float(mae),
    "Eval_Time(s)"         : float(eval_time_seconds),
    "Eval_Speed(s/s)"      : float(eval_speed),
    "Confusion_Matrix"     : cm.tolist(),
    "Classification_Report": report,
}

summary_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

metrics_row = pd.DataFrame([{
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "LoRA_Finetune_BF16",
    "Model"                : MODEL_NAME,
    "LoRA_r"               : LORA_R,
    "LoRA_alpha"           : LORA_ALPHA,
    "Train_Epochs"         : TRAIN_EPOCHS,
    "Effective_Batch_Size" : BATCH_SIZE * GRAD_ACCUM,
    "Learning_Rate"        : LR,
    "Train_Time(s)"        : train_time,
    "Batch_Size"           : INFER_BATCH_SIZE,
    "Accuracy"             : accuracy,
    "Macro_F1"             : macro_f1,
    "Valid_Prediction_Rate": valid_prediction_rate,
    "Strict_Format_Rate"   : strict_format_rate,
    "Off_By_1_Acc"         : np.nan,
    "Adj_Error_Ratio"      : np.nan,
    "Mid_Confusion_Ratio"  : np.nan,
    "MSE"                  : mse,
    "MAE"                  : mae,
    "Eval_Time(s)"         : eval_time_seconds,
    "Eval_Speed(s/s)"      : eval_speed,
}])

metrics_csv_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_metrics.csv")
metrics_row.to_csv(metrics_csv_path, index=False, encoding="utf-8")

print("Saved summary JSON to:", summary_path)
print("Saved metrics CSV to :", metrics_csv_path)
metrics_row

In [ ]:
# ============================================
# Confusion matrix
# ============================================
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=DISPLAY_LABELS, yticklabels=DISPLAY_LABELS,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix - {RUN_ID}")
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_confusion_matrix.png")
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved confusion matrix to:", fig_path)

In [ ]:
# ============================================
# Final console summary
# ============================================
print(f"Run_ID               : {RUN_ID}")
print(f"Task_Type            : {TASK_TYPE}")
print(f"Mode                 : LoRA_Finetune_BF16")
print(f"Model                : {MODEL_NAME}")
print(f"LoRA_r               : {LORA_R}")
print(f"LoRA_alpha           : {LORA_ALPHA}")
print(f"Train_Epochs         : {TRAIN_EPOCHS}")
print(f"Effective_Batch_Size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Learning_Rate        : {LR}")
print(f"Train_Time(s)        : {train_time:.2f}")
print(f"Accuracy             : {accuracy:.4f}")
print(f"Macro_F1             : {macro_f1:.4f}")
print(f"Valid_Prediction_Rate: {valid_prediction_rate:.4f}")
print(f"Strict_Format_Rate   : {strict_format_rate:.4f}")
print(f"MSE                  : {mse:.6f}")
print(f"MAE                  : {mae:.6f}")
print(f"Eval_Time(s)         : {eval_time_seconds:.2f}")
print(f"Eval_Speed(s/s)      : {eval_speed:.4f}")